# 03 — Statistical Tests and Confidence Intervals

Run **after** `dbt build`. Validates churn drivers using:

1. Two-proportion z-test — auto-renew vs churn
2. Chi-square — discount band vs churn
3. Chi-square — registration channel vs churn
4. Mann-Whitney U + bootstrap CI — engagement drop vs churn

Outputs: writes `analytics.mart_statistical_tests` back to DuckDB for Tableau Dashboard 6.

In [ ]:
from pathlib import Path
import sys
import subprocess
import duckdb
import numpy as np
import pandas as pd

# Self-heal: if notebook is attached to a wrong kernel, install missing stats packages
# into the currently running interpreter to avoid ModuleNotFoundError.
def _ensure_pkg(module_name: str, pip_name: str | None = None) -> None:
    try:
        __import__(module_name)
    except ModuleNotFoundError:
        pkg = pip_name or module_name
        print(f"Installing missing package: {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

_ensure_pkg("scipy")
_ensure_pkg("statsmodels")

from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

DUCKDB_PATH = Path('../data/processed/kkbox.duckdb')
rng = np.random.default_rng(42)
print("Python interpreter:", sys.executable)

In [ ]:
con = duckdb.connect(str(DUCKDB_PATH))
df = con.execute('SELECT * FROM analytics.mart_user_churn_features').df()
print('Loaded', df.shape[0], 'users')

## 1. Auto-renew vs churn — two-proportion z-test

In [ ]:
g_on  = df[df['latest_is_auto_renew'] == 1]
g_off = df[df['latest_is_auto_renew'] == 0]

n_on,  k_on  = len(g_on),  g_on['is_churn'].sum()
n_off, k_off = len(g_off), g_off['is_churn'].sum()
p_on,  p_off = k_on / n_on, k_off / n_off

z, p = proportions_ztest([k_on, k_off], [n_on, n_off])
ci_on  = proportion_confint(k_on,  n_on,  alpha=0.05, method='wilson')
ci_off = proportion_confint(k_off, n_off, alpha=0.05, method='wilson')

# Cohen's h effect size
phi1, phi2 = 2*np.arcsin(np.sqrt(p_on)), 2*np.arcsin(np.sqrt(p_off))
h = phi1 - phi2

test_1 = {
    'test_name': 'two_proportion_z_auto_renew',
    'group_a': 'auto_renew_off',
    'group_b': 'auto_renew_on',
    'metric_a': p_off,
    'metric_b': p_on,
    'metric_diff': p_off - p_on,
    'metric_a_ci_low':  ci_off[0],
    'metric_a_ci_high': ci_off[1],
    'metric_b_ci_low':  ci_on[0],
    'metric_b_ci_high': ci_on[1],
    'p_value': float(p),
    'effect_size': float(h),
    'effect_size_type': 'cohen_h',
    'n_a': n_off,
    'n_b': n_on,
    'interpretation': 'Auto-renew off users churn at significantly higher rate.' if p < 0.05 else 'No significant difference.'
}
test_1

## 2. Discount band vs churn — chi-square

In [ ]:
df['discount_band'] = pd.cut(df['avg_discount_rate'].fillna(0),
                              bins=[-0.01, 0.05, 0.20, 0.40, 1.0],
                              labels=['none', 'low', 'medium', 'heavy'])
ct_disc = pd.crosstab(df['discount_band'], df['is_churn'])
chi2, p_disc, dof, expected = stats.chi2_contingency(ct_disc)
n_total = ct_disc.values.sum()
cramer_v = np.sqrt(chi2 / (n_total * (min(ct_disc.shape) - 1)))

test_2 = {
    'test_name': 'chi_square_discount_band',
    'group_a': 'discount_band',
    'group_b': 'is_churn',
    'metric_a': chi2,
    'metric_b': dof,
    'metric_diff': None,
    'metric_a_ci_low': None, 'metric_a_ci_high': None,
    'metric_b_ci_low': None, 'metric_b_ci_high': None,
    'p_value': float(p_disc),
    'effect_size': float(cramer_v),
    'effect_size_type': 'cramers_v',
    'n_a': int(n_total),
    'n_b': int(n_total),
    'interpretation': 'Discount band and churn are dependent.' if p_disc < 0.05 else 'No significant association.'
}
print(ct_disc)
test_2

## 3. Registration channel vs churn — chi-square

In [ ]:
ct_chan = pd.crosstab(df['registered_via'], df['is_churn'])
ct_chan = ct_chan[ct_chan.sum(axis=1) >= 30]  # drop tiny channels
chi2_c, p_c, dof_c, _ = stats.chi2_contingency(ct_chan)
n_total_c = ct_chan.values.sum()
cramer_v_c = np.sqrt(chi2_c / (n_total_c * (min(ct_chan.shape) - 1)))

test_3 = {
    'test_name': 'chi_square_registration_channel',
    'group_a': 'registered_via',
    'group_b': 'is_churn',
    'metric_a': chi2_c,
    'metric_b': dof_c,
    'metric_diff': None,
    'metric_a_ci_low': None, 'metric_a_ci_high': None,
    'metric_b_ci_low': None, 'metric_b_ci_high': None,
    'p_value': float(p_c),
    'effect_size': float(cramer_v_c),
    'effect_size_type': 'cramers_v',
    'n_a': int(n_total_c),
    'n_b': int(n_total_c),
    'interpretation': 'Registration channel and churn are dependent.' if p_c < 0.05 else 'No significant association.'
}
test_3

## 4. Engagement drop vs churn — Mann-Whitney U + bootstrap CI

In [ ]:
ed = df.dropna(subset=['engagement_drop_w15'])
x_churn = ed[ed['is_churn'] == 1]['engagement_drop_w15'].values
x_keep  = ed[ed['is_churn'] == 0]['engagement_drop_w15'].values

u, p_u = stats.mannwhitneyu(x_churn, x_keep, alternative='less')

def bootstrap_diff(a, b, n=1000, alpha=0.05, rng=rng):
    diffs = np.empty(n)
    for i in range(n):
        sa = rng.choice(a, size=len(a), replace=True)
        sb = rng.choice(b, size=len(b), replace=True)
        diffs[i] = np.median(sa) - np.median(sb)
    return float(np.percentile(diffs, 100*alpha/2)), float(np.percentile(diffs, 100*(1-alpha/2)))

ci_low, ci_high = bootstrap_diff(x_churn, x_keep)
median_diff = float(np.median(x_churn) - np.median(x_keep))

test_4 = {
    'test_name': 'mannwhitney_engagement_drop',
    'group_a': 'churned',
    'group_b': 'retained',
    'metric_a': float(np.median(x_churn)),
    'metric_b': float(np.median(x_keep)),
    'metric_diff': median_diff,
    'metric_a_ci_low': None, 'metric_a_ci_high': None,
    'metric_b_ci_low': None, 'metric_b_ci_high': None,
    'p_value': float(p_u),
    'effect_size': median_diff,
    'effect_size_type': 'median_diff_bootstrap',
    'n_a': len(x_churn),
    'n_b': len(x_keep),
    'interpretation': f'Median engagement drop CI95 = ({ci_low:.3f}, {ci_high:.3f})'
}
test_4

## Persist results to DuckDB

In [ ]:
results = pd.DataFrame([test_1, test_2, test_3, test_4])
con.execute('CREATE SCHEMA IF NOT EXISTS analytics')
con.register('results_df', results)
con.execute('CREATE OR REPLACE TABLE analytics.mart_statistical_tests AS SELECT * FROM results_df')
print('Wrote analytics.mart_statistical_tests:', len(results), 'rows')
results

In [ ]:
con.close()
print('Done.')